# BLR (KIA) Airline KPI Rankings — Visual Dashboard

Seaborn charts ranking airlines using the five core KPIs from `v_kpi_metrics_unified`.

**Prerequisite:** Run `kpi_analysis.ipynb` (or apply `sql/99_kpi_metrics_unified.sql`) first.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')

CONN = dict(
    host=os.environ['DB_HOST'],
    port=int(os.environ['DB_PORT']),
    dbname=os.environ['DB_NAME'],
    user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
)

CORE_METRICS = [
    'on_time_arrival_pct',
    'on_time_departure_pct',
    'avg_arrival_delay_min',
    'avg_departure_delay_min',
    'airline_composite_score',
    'airline_rank',
]

WINDOW_ORDER = ['24h', '7d', '30d']
DEFAULT_WINDOW = '30d'
TOP_N = 15

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.05)
PALETTE = sns.color_palette('viridis', n_colors=TOP_N)
FIG_DPI = 110

plt.rcParams.update({
    'figure.dpi': FIG_DPI,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

In [ ]:
def query_df(sql: str, params=None) -> pd.DataFrame:
    with psycopg2.connect(**CONN) as conn:
        return pd.read_sql_query(sql, conn, params=params)


raw = query_df(
    """
    SELECT airline_iata, metric_name, metric_value, window_type, calculated_at
    FROM v_kpi_metrics_unified
    WHERE metric_name = ANY(%(metrics)s)
    """,
    params={'metrics': CORE_METRICS},
)

flight_counts = query_df(
    """
    SELECT
        airline_iata,
        COUNT(*) AS flight_count
    FROM flight_kpi_base
    GROUP BY airline_iata
    ORDER BY flight_count DESC
    """
)

raw['window_type'] = pd.Categorical(raw['window_type'], categories=WINDOW_ORDER, ordered=True)
print(f'Loaded {len(raw):,} KPI rows · {raw["airline_iata"].nunique()} airlines')

In [ ]:
# Wide format: one row per airline × window
kpi = (
    raw.pivot_table(
        index=['airline_iata', 'window_type'],
        columns='metric_name',
        values='metric_value',
        aggfunc='first',
    )
    .reset_index()
)

kpi = kpi.merge(flight_counts, on='airline_iata', how='left')
kpi['airline_label'] = kpi['airline_iata'] + ' (' + kpi['flight_count'].fillna(0).astype(int).astype(str) + ' flts)'

def window_df(window: str = DEFAULT_WINDOW) -> pd.DataFrame:
    df = kpi[kpi['window_type'] == window].copy()
    df = df.dropna(subset=['airline_composite_score'])
    df = df.sort_values('airline_composite_score', ascending=False)
    df['rank'] = np.arange(1, len(df) + 1)
    return df

df30 = window_df('30d')
top = df30.head(TOP_N).copy()
top['airline_label'] = pd.Categorical(top['airline_label'], categories=top['airline_label'][::-1], ordered=True)

df30.head(8)

## 1. Composite score ranking (primary leaderboard)

Score = 40% on-time arrival + 30% on-time departure + 30% delay component. Bar length = composite score; color = on-time arrival %.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

norm = plt.Normalize(top['on_time_arrival_pct'].min(), top['on_time_arrival_pct'].max())
colors = plt.cm.RdYlGn(norm(top['on_time_arrival_pct'].fillna(0)))

bars = ax.barh(
    top['airline_label'],
    top['airline_composite_score'],
    color=colors,
    edgecolor='#333333',
    linewidth=0.4,
    height=0.72,
)

for bar, (_, row) in zip(bars, top.iterrows()):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"#{int(row['rank'])}  {row['airline_composite_score']:.1f}",
        va='center',
        fontsize=9,
        color='#222222',
    )

sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.03)
cbar.set_label('On-time arrival %', fontsize=10)

ax.set_xlabel('Composite KPI score (higher = better)')
ax.set_title(f'Top {TOP_N} airlines at BLR — {DEFAULT_WINDOW} window\n(color = on-time arrival rate)')
ax.set_xlim(0, max(top['airline_composite_score']) * 1.12)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()

## 2. KPI profile heatmap (top airlines)

Normalized view of on-time % and average delays for quick comparison.

In [ ]:
heat_cols = [
  'on_time_arrival_pct',
  'on_time_departure_pct',
  'avg_arrival_delay_min',
  'avg_departure_delay_min',
  'airline_composite_score',
]
heat_labels = [
  'On-time arrival %',
  'On-time departure %',
  'Avg arrival delay (min)',
  'Avg departure delay (min)',
  'Composite score',
]

heat = top.set_index('airline_iata')[heat_cols].copy()
heat_display = heat.copy()
heat_display.columns = heat_labels

# Z-score per column for color (delays inverted so green = good)
z = heat_display.apply(lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) else 0)
z['Avg arrival delay (min)'] *= -1
z['Avg departure delay (min)'] *= -1

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    z,
    annot=heat_display.round(1),
    fmt='',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Relative performance (green = better)'},
    ax=ax,
)
ax.set_title(f'KPI heatmap — top {TOP_N} airlines ({DEFAULT_WINDOW})')
ax.set_ylabel('Airline (IATA)')
ax.set_xlabel('')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 3. On-time performance: arrival vs departure

Grouped bars — arrival KPI is weighted highest in the composite score.

In [ ]:
melt_otp = top.melt(
    id_vars=['airline_iata', 'airline_label'],
    value_vars=['on_time_arrival_pct', 'on_time_departure_pct'],
    var_name='metric',
    value_name='pct',
)
melt_otp['metric'] = melt_otp['metric'].map({
    'on_time_arrival_pct': 'On-time arrival',
    'on_time_departure_pct': 'On-time departure',
})

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=melt_otp,
    x='airline_iata',
    y='pct',
    hue='metric',
    palette={'On-time arrival': '#2ca02c', 'On-time departure': '#1f77b4'},
    edgecolor='#333',
    linewidth=0.3,
    ax=ax,
)
ax.axhline(15, color='#888', linestyle=':', linewidth=1, label='15 min OTP threshold (reference)')
ax.set_ylabel('On-time rate (%)')
ax.set_xlabel('Airline (IATA)')
ax.set_title(f'On-time departure vs arrival — top {TOP_N} airlines ({DEFAULT_WINDOW})')
ax.set_ylim(0, 105)
plt.xticks(rotation=45, ha='right')
ax.legend(title='', loc='lower right')
sns.despine()
plt.tight_layout()
plt.show()

## 4. Delay profile (minutes)

Negative values = early; positive = late.

In [ ]:
melt_delay = top.melt(
    id_vars=['airline_iata'],
    value_vars=['avg_arrival_delay_min', 'avg_departure_delay_min'],
    var_name='metric',
    value_name='minutes',
)
melt_delay['metric'] = melt_delay['metric'].map({
    'avg_arrival_delay_min': 'Arrival delay',
    'avg_departure_delay_min': 'Departure delay',
})

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=melt_delay,
    x='airline_iata',
    y='minutes',
    hue='metric',
    palette={'Arrival delay': '#d62728', 'Departure delay': '#ff7f0e'},
    edgecolor='#333',
    linewidth=0.3,
    ax=ax,
)
ax.axhline(0, color='#444', linewidth=1)
ax.axhline(15, color='#888', linestyle='--', linewidth=1, alpha=0.8)
ax.text(0.01, 15.5, '15 min late', transform=ax.get_xaxis_transform(), fontsize=8, color='#666')
ax.set_ylabel('Average delay (minutes)')
ax.set_xlabel('Airline (IATA)')
ax.set_title(f'Average delays — top {TOP_N} airlines ({DEFAULT_WINDOW})')
plt.xticks(rotation=45, ha='right')
sns.despine()
plt.tight_layout()
plt.show()

## 5. Performance map: on-time arrival vs average arrival delay

Ideal quadrant: **high on-time %, low delay**. Point size = flight volume at BLR.

In [ ]:
scatter = df30.dropna(subset=['on_time_arrival_pct', 'avg_arrival_delay_min']).copy()

fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(
    data=scatter,
    x='avg_arrival_delay_min',
    y='on_time_arrival_pct',
    size='flight_count',
    sizes=(40, 400),
    hue='airline_composite_score',
    palette='viridis',
    alpha=0.85,
    edgecolor='#333',
    linewidth=0.4,
    legend='brief',
    ax=ax,
)

for _, row in scatter.nlargest(12, 'airline_composite_score').iterrows():
    ax.annotate(
        row['airline_iata'],
        (row['avg_arrival_delay_min'], row['on_time_arrival_pct']),
        textcoords='offset points',
        xytext=(4, 4),
        fontsize=8,
        alpha=0.9,
    )

ax.axhline(scatter['on_time_arrival_pct'].median(), color='#aaa', linestyle=':', linewidth=1)
ax.axvline(15, color='#c44', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(0, color='#888', linestyle='-', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Average arrival delay (minutes)')
ax.set_ylabel('On-time arrival rate (%)')
ax.set_title(f'Airline performance map — all ranked airlines ({DEFAULT_WINDOW})')
handles, labels = ax.get_legend_handles_labels()
if handles:
    ax.legend(title='Composite score', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Ranking stability across time windows

Composite score by airline for 24h / 7d / 30d (top airlines only).

In [ ]:
top_codes = top['airline_iata'].tolist()
trend = kpi[
    (kpi['airline_iata'].isin(top_codes))
    & kpi['airline_composite_score'].notna()
].copy()

fig, ax = plt.subplots(figsize=(12, 7))
sns.lineplot(
    data=trend,
    x='window_type',
    y='airline_composite_score',
    hue='airline_iata',
    marker='o',
    linewidth=2,
    palette='tab20',
    ax=ax,
)
ax.set_xlabel('Aggregation window')
ax.set_ylabel('Composite KPI score')
ax.set_title(f'Composite score trend — top {TOP_N} airlines')
ax.legend(title='Airline', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
sns.despine()
plt.tight_layout()
plt.show()

## 7. Faceted leaderboard — all three windows

In [ ]:
facet_rows = []
for w in WINDOW_ORDER:
    wdf = window_df(w).head(TOP_N)
    wdf['window_type'] = w
    facet_rows.append(wdf)
facet_df = pd.concat(facet_rows, ignore_index=True)

g = sns.FacetGrid(
    facet_df,
    col='window_type',
    col_order=WINDOW_ORDER,
    height=6,
    aspect=0.85,
    sharex=False,
)


def _facet_bar(data, color, **kwargs):
    data = data.sort_values('airline_composite_score')
    ax = plt.gca()
    sns.barplot(
        data=data,
        y='airline_iata',
        x='airline_composite_score',
        hue='on_time_arrival_pct',
        dodge=False,
        palette='RdYlGn',
        edgecolor='#333',
        linewidth=0.3,
        ax=ax,
        legend=False,
    )
    ax.set_ylabel('')
    ax.set_xlabel('Composite score')


g.map_dataframe(_facet_bar)
g.set_titles(col_template='{col_name} window')
g.fig.suptitle(f'Airline composite ranking — top {TOP_N} per window', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 8. Executive summary dashboard (single figure)

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2, hspace=0.32, wspace=0.28)

# A: composite bars
ax1 = fig.add_subplot(gs[0, 0])
plot_top = top.sort_values('airline_composite_score')
sns.barplot(
    data=plot_top,
    y='airline_iata',
    x='airline_composite_score',
    hue='on_time_arrival_pct',
    dodge=False,
    palette='RdYlGn',
    ax=ax1,
    legend=False,
)
ax1.set_title('Composite ranking')
ax1.set_xlabel('Score')
ax1.set_ylabel('Airline')

# B: on-time grouped
ax2 = fig.add_subplot(gs[0, 1])
sns.barplot(data=melt_otp, x='airline_iata', y='pct', hue='metric', ax=ax2,
            palette={'On-time arrival': '#2ca02c', 'On-time departure': '#1f77b4'})
ax2.set_title('On-time %')
ax2.set_xlabel('')
ax2.set_ylabel('%')
ax2.tick_params(axis='x', rotation=45)
ax2.legend(fontsize=8, loc='lower right')

# C: delays
ax3 = fig.add_subplot(gs[1, 0])
sns.barplot(data=melt_delay, x='airline_iata', y='minutes', hue='metric', ax=ax3,
            palette={'Arrival delay': '#d62728', 'Departure delay': '#ff7f0e'})
ax3.axhline(0, color='#444', lw=0.8)
ax3.set_title('Avg delay (min)')
ax3.set_xlabel('')
ax3.tick_params(axis='x', rotation=45)
ax3.legend(fontsize=8)

# D: scatter
ax4 = fig.add_subplot(gs[1, 1])
sns.scatterplot(
    data=scatter.head(25),
    x='avg_arrival_delay_min',
    y='on_time_arrival_pct',
    size='flight_count',
    sizes=(30, 200),
    hue='airline_composite_score',
    palette='viridis',
    ax=ax4,
    legend=True,
)
ax4.axvline(15, color='#c44', ls='--', lw=1, alpha=0.7)
ax4.set_title('Arrival OTP vs delay')
ax4.set_xlabel('Avg arrival delay (min)')
ax4.set_ylabel('On-time arrival %')

fig.suptitle(
    f'BLR (KIA) Airline KPI Dashboard — {DEFAULT_WINDOW} window · top {TOP_N} airlines',
    fontsize=15,
    fontweight='bold',
    y=1.01,
)
plt.tight_layout()
plt.show()

## Export ranked table (CSV)

In [ ]:
export_path = Path('..') / 'airline_kpi_ranking_30d.csv'
cols = [
    'rank', 'airline_iata', 'flight_count',
    'airline_composite_score', 'on_time_arrival_pct', 'on_time_departure_pct',
    'avg_arrival_delay_min', 'avg_departure_delay_min',
]
df30[cols].to_csv(export_path, index=False)
print(f'Saved: {export_path.resolve()}')